## Import all what we need

In [1]:
import os
import math
import glob
import time
import numba
import requests
import numpy as np
import pandas as pd
import numexpr as ne
from numba import njit
from tqdm import tqdm
from tqdm.notebook import tqdm as tqdm_nb
from io import StringIO

In [2]:
# Check the number of this computer's CPUs.
import multiprocessing
cup_num = multiprocessing.cpu_count()
print(f"CPU number: {cup_num}")

from pandarallel import pandarallel
pandarallel.initialize(nb_workers=4,progress_bar=False)    # Initialize.
#pandarallel.initialize(nb_workers=8,progress_bar=True)    # Initialize.


CPU number: 256
INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [3]:
# Scale the pandas workflow with modin on computer engine ray.
#import ray
#os.environ["MODIN_ENGINE"] = "ray"
#ray.init(num_cpus = 4, ignore_reinit_error=True)
#import modin.pandas as pd

## File Paths


<table><tr><td bgcolor=skyblue><font size=24> Could be changed ! </font></td></tr></table>

In [4]:
#########################################################
def_path = '/home/jingxin/data/def/'
url_path = '/home/jingxin/data/url/'
unc_path = '/home/jingxin/ExoMolHR/ExoMolHR_list.csv'
database_path = '/mnt/data/exomol/exomol3_data/'
#database_path = '/home/jingxin/data/exomol_data/'
loc_result_path = '/home/jingxin/data/exomolhr/loc_result/'    # Create a folder for saving local format result files.
web_result_path = '/home/jingxin/data/exomolhr/web_result/'    # Create a folder for saving web format result files.
web_log_path = 'home/jingxin/data/exomolhr/web_log/'           # Create a folder for saving log file.

T = 300
molecule = 'H2O'
isotopologue = '1H2-16O'
dataset = 'POKAZATEL'
min_v = 0.00
max_v = 1000.00
max_uncertainty = 0.01
min_intensity = 10E-30
#########################################################

## Report time

In [5]:
class Timer:    
    def start(self):
        self.start_CPU = time.process_time()
        self.start_sys = time.time()
        return self

    def end(self, *args):
        self.end_CPU = time.process_time()
        self.end_sys = time.time()
        self.interval_CPU = self.end_CPU - self.start_CPU
        self.interval_sys = self.end_sys - self.start_sys
        print('{:25s} : {}'.format('Running time on CPU', self.interval_CPU), 's')
        print('{:25s} : {}'.format('Running time on system', self.interval_sys), 's')

# Part 1: Get Molecule, Isotopologue, Dataset and Abundance

Get the names of molecule name, isotopologue name and dataset name from the api__urls.txt which saved the URLs with molecule, isotopologue and dataset. Combine them with '/' for reading files from folders more convenient later.

In [43]:
def mol_param(unc_path):
    colnames=['id','molecule','isotopologue','isoformula','dataset','abundance','main format','qn label','qn format']
    molparam_df = pd.read_csv(unc_path, usecols=[0,1,2,3,4,5,7,8,9], names=colnames, header=0)     
    molparam = molparam_df[molparam_df['molecule'].isin([molecule]) & 
                           molparam_df['isotopologue'].isin([isotopologue]) &
                           molparam_df['dataset'].isin([dataset])].values[0]
    isoformula = molparam[3]
    abundance = molparam[5]
    J_format = molparam[6].split(',')[3]
    qns_label = molparam[7]
    qns_format = molparam[8]
    mol_iso_ds_path = molecule + '/' + isotopologue + '/' + dataset
    print('Molecule \t\t\t:', molecule)
    print('Isotopologue \t\t\t:', isotopologue) 
    print('Isotopologue formula \t\t:', isoformula) 
    print('Dataset \t\t\t:', dataset)  
    print('Abundance \t\t\t:', abundance)   
    print('J format \t\t\t:', J_format)
    print('Quantumn number labels \t\t:', qns_label)
    print('Quantumn number formats \t:', qns_format, '\n')    
    return(molparam_df, isoformula, abundance, J_format, qns_label, qns_format, mol_iso_ds_path)

In [44]:
molparam_df, isoformula, abundance, J_format, qns_label, qns_format, mol_iso_ds_path = mol_param(unc_path)

Molecule 			: H2O
Isotopologue 			: 1H2-16O
Isotopologue formula 		: (1H)2(16O)
Dataset 			: POKAZATEL
Abundance 			: 1
J format 			: %7d
Quantumn number labels 		: Ka,Kc,v1,v2,v3,Grve
Quantumn number formats 	: %2d,%2d,%2d,%2d,%2d,%3s 



In [45]:
molparam_df

,id,molecule,isotopologue,isoformula,dataset,abundance,main format,qn label,qn format
0,1,AlCl,27Al-35Cl,(27Al)(35Cl),YNAT,1,"%12d,%12.6f,%6d,%7d,%12.6f","+/-,e/f,ElecState,v,Lambda,Sigma,Omega","%1s,%1s,%12s,%3d,%3d,%5d,%5d"
1,2,AlCl,27Al-37Cl,(27Al)(37Cl),YNAT,1,"%12d,%12.6f,%6d,%7d,%12.6f","+/-,e/f,ElecState,v,Lambda,Sigma,Omega","%1s,%1s,%12s,%3d,%3d,%5d,%5d"
2,3,AlH,27Al-1H,(27Al)(1H),AloHa,1,"%12d,%12.6f,%6d,%7d,%12.6f","+/-,e/f,ElecState,v,Lambda,Sigma,Omega","%1s,%1s,%12s,%3d,%3d,%5d,%5d"
3,4,AlH,27Al-2H,(27Al)(2H),AloHa,1,"%12d,%12.6f,%6d,%7d,%12.6f","+/-,e/f,ElecState,v,Lambda,Sigma,Omega","%1s,%1s,%12s,%3d,%3d,%5d,%5d"
4,5,AlO,26Al-16O,(26Al)(16O),ATP,1,"%12d,%12.6f,%6d,%7.1f,%12.6f","+/-,e/f,ElecState,v,Lambda,Sigma,Omega","%1s,%1s,%12s,%3d,%3d,%5.1f,%5.1f"
5,6,AlO,27Al-16O,(27Al)(16O),ATP,1,"%12d,%12.6f,%6d,%7.1f,%12.6f","+/-,e/f,ElecState,v,Lambda,Sigma,Omega","%1s,%1s,%12s,%3d,%3d,%5.1f,%5.1f"
6,7,AlO,27Al-17O,(27Al)(17O),ATP,1,"%12d,%12.6f,%6d,%7.1f,%12.6f","+/-,e/f,ElecState,v,Lambda,Sigma,Omega","%1s,%1s,%12s,%3d,%3d,%5.1f,%5.1f"
7,8,AlO,27Al-18O,(27Al)(18O),ATP,1,"%12d,%12.6f,%6d,%7.1f,%12.6f","+/-,e/f,ElecState,v,Lambda,Sigma,Omega","%1s,%1s,%12s,%3d,%3d,%5.1f,%5.1f"
8,9,C2,12C2,(12C)2,8states,1,"%12d,%12.6f,%6d,%7d,%12.6f","+/-,e/f,ElecState,v,Lambda,Sigma,Omega","%1s,%1s,%12s,%3d,%3d,%5d,%5d"
9,10,C2H2,12C2-1H2,(12C)2(1H)2,aCeTY,1,"%12d,%12.6f,%6d,%7d,%12.6f","Gtot,K,e/f,Grot","%3s,%2d,%1s,%3s"


# Part 2: Process Data

## 2.1 Read Partition Function File

In [46]:
# Read partition function with online webpage.
def read_web_pf(T, mol_iso_ds_path):
    '''Get partition function from ExoMol website directly.'''    
    pf_url = ('https://exomol.com/db/' + mol_iso_ds_path + '/' + isotopologue + '__' + dataset + '.pf')
    pf_content = requests.get(pf_url).text
    pf_col_name = ['T', 'Q']
    print('Read the partition function file.')
    pf_df = pd.read_csv(StringIO(pf_content), sep='\\s+', names=pf_col_name, header=None)
    max_T = pf_df.count()[0]
    inNumberint = int(T)
    if T != inNumberint:
        raise Exception('Sorry, please type an integer as temperature.')
    elif T > max_T:
        print('The maximum temperature is', max_T, 'K.')
        raise Exception('Sorry, please type a smaller T.')
    elif T < 1:
        raise('Sorry, please type a new T which is larger than 0.')
    else:
        Q = pf_df['Q'][T-1]
        print('The partition function at T =', T, 'K is', Q, '\n')
    return(Q)


# Read partition function with local partition function file.
def read_exomol_pf(T, database_path):
    pf_filename = (database_path + mol_iso_ds_path + '/' + isotopologue + '__' + dataset + '.pf')
    pf_col_name = ['T', 'Q']
    print('Read the partition function file.')
    pf_df = pd.read_csv(pf_filename, sep='\\s+', names=pf_col_name, header=None)
    max_T = pf_df.count()[0]
    inNumberint = int(T)
    if T != inNumberint:
        raise Exception('Sorry, please type an integer as temperature.')
    elif T > max_T:
        print('The maximum temperature is', max_T, 'K.')
        raise Exception('Sorry, please type a smaller T.')
    elif T < 1:
        raise('Sorry, please type a new T which is larger than 0.')
    else:
        Q = pf_df['Q'][T-1]
        print('The partition function at T =', T, 'K is', Q, '\n')
    return(Q)

## 2.2 Calculating

Calculate intensity at the chosen temperature.

### Constants and Parameters

In [47]:
# Parameters for calculating.
import astropy.constants as ac
h = ac.h.to('J s').value          # Planck's const (J s)
c = ac.c.to('cm/s').value         # Velocity of light (cm s^{-1})
kB = ac.k_B.to('J/K').value       # Boltzmann's const (J K^{-1})
c2 = h * c / kB                   # Second radiation constant (cm K)
c2_T = - c2 / T                   # - c2 / T (cm)
pi_c_8 = 1 / (8 * np.pi * c)      # 8 * pi * c (cm-1 s)

In [48]:
# @njit(parallel=True, fastmath=True)
# def cal_intensity(A, Epp, gp, Q, v, c2_T, pi_c_8, abundance):
#     I = gp * A * np.exp(c2_T * Epp) * (1 - np.exp(c2_T * v)) * pi_c_8 / (v ** 2) / Q * abundance
#     return(I)

In [49]:
def cal_intensity(A, Epp, gp, Q, v, c2_T, pi_c_8, abundance):
    I = ne.evaluate('gp * A * exp(c2_T * Epp) * (1 - exp(c2_T * v)) * pi_c_8 / (v ** 2) / Q * abundance')
    return(I)

## 2.3 Format

Format the results.

### 2.3.1 Format the Quantumn Numbers

In [60]:
def format_qns(loc_df, qns_label, qns_format):
    qn_label_list = qns_label.split(',')
    qn_format_list = qns_format.split(',')
    label_num = len(qn_label_list)
    qn_format = [format.replace(format[1:-1],str(pd.to_numeric(format[1:-1])+1)).replace("%",'{: >')+'}' for format in qn_format_list]
    qn_label_u_list = [loc_df[qn_label_list[i]+"'"].map(qn_format[i].format) for i in range(label_num)]
    qn_label_l_list = [loc_df[qn_label_list[i]+'"'].map(qn_format[i].format) for i in range(label_num)]

    max_qn_format_num = 50
    max_qn_format = '{: <' + str(max_qn_format_num)+'s}'
    qn_label_u = pd.DataFrame(qn_label_u_list).sum(axis=0).parallel_map(max_qn_format.format)
    qn_label_l = pd.DataFrame(qn_label_l_list).sum(axis=0).parallel_map(max_qn_format.format)
    return(qn_label_u, qn_label_l)

### 2.3.2 Format All Results

In [70]:
def format(loc_df, qns_label, qns_format, molecule, isoformula, dataset):

    A = pd.to_numeric(loc_df['A']).values
    Epp = pd.to_numeric(loc_df['E"']).values
    gp = pd.to_numeric(loc_df["g'"]).values
    v = pd.to_numeric(loc_df['Frequency']).values
    # Q = read_web_pf(T, mol_iso_ds_path)
    Q = read_exomol_pf(T, database_path)
    I = cal_intensity(A, Epp, gp, Q, v, c2_T, pi_c_8, abundance)
    Jfmt = J_format.replace("%",'{: >')+'}'

    qn_label_u, qn_label_l = format_qns(loc_df, qns_label, qns_format)
    web_df = pd.DataFrame()
    web_df['Frequency'] = pd.Series(v).parallel_map('{: >12.6f}'.format)
    web_df['Uncertainty'] = pd.Series(loc_df['Uncertainty'].values).parallel_map('{: >12.6f}'.format)
    web_df['A'] = pd.Series(loc_df['A'].values).parallel_map('{: >10.4E}'.format)
    web_df['Intensity'] = pd.Series(I)
    web_df['Molecule'] = molecule
    web_df['Isotopologue'] = isoformula
    web_df['Dataset'] = dataset
    web_df['E"'] = pd.Series(Epp).parallel_map('{: >12.6f}'.format)
    web_df["g'"] = pd.Series(loc_df["g'"].values).parallel_map('{: >6d}'.format)
    web_df['g"'] = pd.Series(loc_df['g"'].values).parallel_map('{: >6d}'.format)
    web_df["J'"] = pd.Series(loc_df["J'"].values).parallel_map(Jfmt.format)
    web_df['J"'] = pd.Series(loc_df['J"'].values).parallel_map(Jfmt.format)
    web_df["QN'"] = qn_label_u
    web_df['QN"'] = qn_label_l
    web_df = web_df[web_df['Intensity'] > min_intensity]
    web_df['Intensity'] = web_df['Intensity'].parallel_map('{: >10.4E}'.format)
    order = ['Frequency', 'Uncertainty', 'A', 'Intensity', 'Molecule', 'Isotopologue', 'Dataset', 'E"', "g'", 'g"', "J'", 'J"', "QN'", 'QN"']
    web_df = web_df[order]

    return(web_df)

# Part 3: Get Results

Read local results files and process the data to get the ExoMolHR web format result.

In [71]:
def read_loc_get_result(loc_result_path, mol_iso_ds_path, isoformula):
    
    web_df = pd.DataFrame()
    loc_result_filepath = loc_result_path + mol_iso_ds_path.replace('/','__') + '__' + isoformula  + '.csv'
    print('Read the local result file.')
    read_loc = pd.read_csv(loc_result_filepath, header=0, chunksize=100_000_000,
                           iterator=True, low_memory=False)
    for chunk in read_loc:
        chunk = chunk[pd.to_numeric(chunk['Frequency']).between(min_v, max_v)]
        chunk = chunk[pd.to_numeric(chunk['Uncertainty']) < max_uncertainty]
        web_format_chunk = format(chunk, qns_label, qns_format, molecule, isotopologue, dataset)
    web_df = pd.concat([web_df, web_format_chunk])
    
    pd.set_option("display.max_columns",30)                           
    return(web_df)

In [72]:
t = Timer()
t.start()

web_df = read_loc_get_result(loc_result_path, mol_iso_ds_path, isoformula)
web_df.to_csv(web_result_path + mol_iso_ds_path.replace('/','__') + '__' + isoformula + '_web.csv', header=True, index=False)
print('The web format result has been saved.')
    
t.end()

Read the local result file.
Read the partition function file.
The partition function at T = 300 K is 178.1206 

The web format result has been saved.
Running time on CPU       : 40.728267391 s
Running time on system    : 42.9453125 s


In [73]:
web_df

,Frequency,Uncertainty,A,Intensity,Molecule,Isotopologue,Dataset,"E""",g',"g""",J',"J""",QN',"QN"""
0,0.400554,0.000002,9.3492E-10,2.3927E-28,H2O,1H2-16O,POKAZATEL,1907.615571,27,21,4,3,2 3 0 1 0 B1 ...,3 0 0 1 0 B2 ...
1,0.741682,0.000000,1.9604E-09,4.3202E-25,H2O,1H2-16O,POKAZATEL,446.510665,39,33,6,5,1 6 0 0 0 B2 ...,2 3 0 0 0 B1 ...
2,0.895095,0.000003,7.9254E-09,3.8213E-28,H2O,1H2-16O,POKAZATEL,2129.598921,33,27,5,4,3 2 0 1 0 B2 ...,4 1 0 1 0 B1 ...
9,3.210931,0.000002,4.6927E-07,1.7371E-27,H2O,1H2-16O,POKAZATEL,2126.407583,9,11,4,5,4 0 0 1 0 A1 ...,3 3 0 1 0 A2 ...
15,4.002629,0.000003,1.4774E-06,1.5559E-26,H2O,1H2-16O,POKAZATEL,1739.483705,5,7,2,3,2 0 0 1 0 A1 ...,1 3 0 1 0 A2 ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
442721,998.508058,0.000139,6.8794E-02,1.6419E-28,H2O,1H2-16O,POKAZATEL,3770.724167,23,21,11,10,10 2 0 1 0 A1 ...,7 3 0 1 0 A2 ...
442722,998.508941,0.000011,1.2704E-03,1.5287E-28,H2O,1H2-16O,POKAZATEL,3244.599300,93,93,15,15,1 14 0 1 0 B2 ...,4 11 0 0 0 B1 ...
442732,998.521538,0.000139,6.8793E-02,4.9258E-28,H2O,1H2-16O,POKAZATEL,3770.710693,69,63,11,10,10 1 0 1 0 B1 ...,7 4 0 1 0 B2 ...
442866,998.809363,0.000002,1.5644E-02,2.3640E-24,H2O,1H2-16O,POKAZATEL,1631.382967,51,57,8,9,3 6 0 1 0 B2 ...,6 3 0 0 0 B1 ...


In [17]:
#max_qn_format_num = 50
#max_qn_format= '%'+str(max_qn_format_num)+'s'
#qn_format_withoutspaces = ''.join(states_qn_format)
#qn_format = max_qn_format % qn_format_withoutspaces